### 项目经历
**你说使用Trino每10分钟upsert到Iceberg，能详细解释一下Iceberg的upsert底层是如何实现的吗？涉及到哪些文件操作？**

upsert是通过merge into来实现的，简单来说，当你在 Trino 中执行一个 UPSERT 操作时，在MOR模式下，Iceberg 并不会直接修改或覆盖已有的数据文件。相反，它会生成一个新的“删除文件”来标记哪些旧数据需要被逻辑删除，同时写入一个包含新数据的数据文件。在后续查询时，引擎会动态地将数据文件和删除文件合并，呈现出更新后的结果。核心原理是利用MOR模式的位置删除，我举个例子，比如说Merge into识别到某个user_id需要更新，它不会直接在源文件的基础上修改，它会追加删除文件，里面记录了需要删除的文件和行号两列，再追加一个数据文件，里面记录了更新后的内容，查询时引擎会把这三类文件合并在一起，并呈现出最终正确的数据。除此之外，Iceberg还支持位置删除，虽然我没有用过，但我知道大概的原理，它和位置删除类似，它也需要先追加删除文件，但这个删除文件不记录具体的行号，它记录了要删除的条件，但它不会去具体的文件中找到要删除哪一行，下游计算引擎在读取时再去找，优势是写入比位置删除更快，劣势是读取更慢。然后是Iceberg的v3格式，它引入了删除向量，这是对位置删除的一种增强，这种删除模式将不会在数据文件中追加删除文件了，它会在清单文件中维护一个删除向量的数组，通过稀疏数组和压缩的方式，使其占用的空间极小，好处是和位置删除一样能定位到具体删除的行，同时不必像位置删除一样要维护大量的删除文件，同时下游计算引擎合并数据文件和删除文件时也会更快。

**你提到将单表文件数量维持在1000左右，具体的维护任务是怎么设计的？compaction的频率和策略是什么？**

我是通过分级维护的，分为三级维护。第一级，写入后即维护，我的所有写入任务都是在Airflow中的，我会在写入完成后立刻合并最近写入的文件，因为涉及的文件数量极少，维护速度非常快，但是即使它非常快，我们也不能等它维护完成才继续跑下游任务，因此我的Airflow下游dag只依赖写入任务，不依赖维护任务，只要写入任务完成，下游就可以继续运行了，让其异步的维护数据文件，然后是第二级，每两小时运行一次，在每两小时的写入完成后，也会异步的合并最近两小时写入的文件，最后是第三级，需要挑选一个业务低峰期，做全量的合并及删除快照和删除孤儿文件以及打标签等操作，也是异步运行。但这些还不够，我还需要优化依赖任务，防止写入任务和维护任务同时运行导致锁争抢报错，我是加了一个Iceberg部分维护的参数，这个参数能控制维护任务部分成功，如果遇到写入任务了，就自动发起维护重试。但说实话，我的维护策略是基于时间的，虽然实现了使表保持在一个比较好的性能，但有点笨重，实现也不够优雅，我认为真正智能的维护是需要能实时感知到表的状态，自动在后台发起维护任务，我也了解到像Apache Amore这样的技术可以实现这种效果，但在我的项目中，我就遵循够用即可的原则，同时考虑到Apache Amore的成熟度和生态兼容性问题，就没有引入Apache Amore。

**Iceberg的快照机制是如何工作的？在实际业务中有没有用到时间旅行功能？**

Iceberg每次写入都会都会生成一个快照，它是不可变的，快照之间通过增量快照链关联，支持时间旅行查询，查询时通过快照 ID 或时间戳访问历史数据，我在实际业务中用时间旅行代替了ODS的全量快照，不需要用大量的存储去支撑历史数据的审计了，只需要维护清单元数据即可，也代替了周期快照事实表，不再需要为类似购物车状态每天维护一份数据了，然后是维度表，我只需要在维度表中保存最新的状态即可，不再需要每天全量快照或者建拉链表做复杂的维护了，这使得存储成本至少下降了100倍吧，但是快照机制也不是万能的，如果维护不当可能会造成Iceberg表性能暴跌，比如说在高频upsert的情况下，每次写入都会生成好几个清单文件和清单列表，再不加以维护的情况下几个月表就变得不可用了，我的维护策略是每天为表打一个标签，然后删除表的所有快照，这样可以每天只保留一份快照，但这个策略不是固定的，具体业务需要具体分析，比如某些业务就是要追溯每天每10分钟的，但不需要追溯历史，所以这个维护策略得跟着业务走。

**Trino vs Spark SQL：在你的架构中，Trino负责10分钟级upsert，Spark负责零点补数据，为什么这样分工？两者的优劣对比是什么？**

我先说一下为什么要用Trino做10分钟级别的upsert，首先我需要保证我每10分钟计算的数据在第12分钟可见，Trino基于全内存的MPP常驻架构先天就适合这种场景，它不需要像Spark SQL一样每启动一个JOB就会经历一分钟的启动加调度延迟，然后不会像Spark SQL一样每个JOB经历优化器重量级且漫长的优化，其次它的阶段之间不会落盘，在内存中直接交换数据即可，这样计算速度会非常快，我曾经试图引入Apache Kyuubi来解决Spark SQL的启动延迟问题，原理是通过常驻Spark引擎实现的，但我碰到一系列的问题，重量级的优化器承载不了高频的SQL提交，很容易把Driver打崩，我不得不规划排队等待逻辑，但是还是解决不了一个重量级SQL一直将Executor打崩，导致小任务也无法运行的情况，Trino的SQL级别的资源限制就能完美解决该问题，即使我通过增加计算资源的方式绕开了这些问题，它还会碰到重量级的优化以及Spark的容错机制使计算变得非常慢的问题。我再讲一下为什么需要用Spark SQL负责零点补数据，正如我刚才所说，Spark SQL的特性导致小任务特别慢，但如果你用它跑大任务，它的好处就体现出来了，它的重量级的优化和容错机制使计算变得非常稳，Trino可能重试很多次都无法成功的任务Spark SQL就能稳定运行。

**Doris应用场景：为什么选择Doris作为ADS层存储？与直接用Iceberg+Trino提供报表服务相比，优势在哪里？**

首先Doris支持存算一体模式，先天就能屏蔽Iceberg + Trino模式下存算分离架构引入的网络延迟，再者Doris支持倒排索引，能精确定位到行，查一行数据只需要扫一行数据即可，虽然存储成本比较高，但ADS层比较少的数据量先天就屏蔽了该问题，反观Iceberg那边支持分区索引、清单索引及parquet page级别的索引，按照默认配置情况下可能读一行数据就需要扫描1MB的parquet page，如果数据比较分散，这个读放大问题还会继续扩大，再然后Doris还为高并发的点查引入了短路径优化和行列混存等一系列优化手段，能支撑的QPS远高于Trino。

**Shuffle优化：你提到未来要引入Celeborn，当前Spark SQL的shuffle瓶颈体现在哪里？预估能带来多少性能提升？**

说实话，在高版本的Spark SQL中，我不认为shuffle有什么瓶颈，它在内部已经把随机读写转化成顺序读写了，也不会发起大规模的网络连接使网络io爆炸，引入Celeborn只能将shuffle数据的行式存储转化为列式存储，减少一点存储的数据量和网络拉取的数据量，但你在写入和读取时却要引入额外的行式转列式的计算，会增加一些cpu消耗，之所以会选择引入Celeborn，是因为我的Spark SQL是运行在k8s上的并开启了动态资源分配，动态资源分配的原理是Driver监控Executor的负载，如果Executor空闲，Driver是希望将它回收掉以节省资源的，但是Driver识别到Executor空闲希望回收时，它又识别到里面有shuffle数据，如果回收了会导致下游拉取失败，进而引发整个job的失败重试，我就不得不开启Shuffle Tracking，它的原理是是给Driver标识该executor虽然已经空闲了，但下游还没有消费完它的shuffle数据，不能回收，所以这个空闲Executor是需要一直等到shuffle数据被消费完之后才能被回收，这样的话空闲Executor又会多占用很长的资源时间，我的架构中Celeborn就是为了解决这个问题的，引入Celeborn是为了让Spark将shuffle数据写入Celeborn，实现计算和存储的解耦，这样Executor能随时被回收，节省大量计算资源。不知道面试官有没有用过抢占式容器实例，在我的设想中，spark on k8s是非常适合运行在抢占式容器实例中的，但前提就是把这个shuffle存储解耦出去，使其变成真正的无状态计算引擎，这样能让计算成本降低100倍不止。

**零点大任务没运行前数据在第12分钟可见，那如果任务失败或者延迟了，有什么降级或补偿机制？**

一般是依赖Airflow的重试机制，如果多次重试都不成功，则发送邮件告警，此时就需要人工介入了，检查的点大概分为是否数据倾斜，数据倾斜的话需要加盐值，如果是资源不足，需申请资源并先紧急暂停一些非核心业务，将资源预留给核心业务，如果是及其复杂的任务，短时间也拿不到相应的资源，同时业务允许的话，可以采取抽样计算的方式，保证大盘可见。

**在Trino查询Iceberg表时，遇到过哪些性能问题？是如何优化的？（提示：分区裁剪、谓词下推等）**

首先是小文件问题，trino在读取时需要读取大量的 Manifest 文件来构建 Split，导致查询还没开始跑，光生成执行计划就耗时很久，但在优化合理的Iceberg表中这种问题比较少见
然后是数据无序导致的“全表扫描”，我举个例子，如果数据是随意写入的，你查一个user_id=123，可能每个parquet文件的min，max索引中都恰好包含了123，导致全表扫描，优化方案有两种，第一种是可以在合并文件对常用字段排序，这样能优化min，max索引的命中率，第二种是是加布隆过滤器，Trino 在查询时会利用 Bloom Filter 快速判断数据是否存在于文件中，特别适合等值查询。
最后是Iceberg的分区裁剪、谓词下推等，这些手段在Trino已经优化得特别好了，很难碰到类似问题，即使碰到了，也可以通过执行计划分析出来，通过调整SQL使其正确裁剪、下推

**Iceberg表维护时，除了文件数量，还会考虑哪些成本因素？（如：小文件合并、过期快照清理等）**

除了文件数量，我还会考虑计算成本，Iceberg的维护操作是一个重量级操作，清理过频会占用大量计算资源，清理过少会使表性能急剧下降，因此我要做的是找到一个点使其达到性能和成本的平衡，我是通过分级的策略来为Iceberg表做维护的，分了小规模合并，中规模合并和大规模合并，如我第二点所说。

**Iceberg有哪几类Catalog？**

Iceberg的REST Catalog分了好几类
Hive Metastore Catalog：传统方案
REST Catalog：这种性能最好，它通过rest api提供服务，不像Hive Metastore Catalog，必须使用Thrift协议才能通信，这限制了很多不支持Thrift协议的应用，我之所以选择Polaris作为catalog，是因为它支持完整的鉴权。
Hadoop Tables：无 Catalog 依赖，直接基于文件系统，但事务控制较弱 
JDBC Catalog：简单易用，但生态不成熟，有单点瓶颈 
Glue Catalog：国外生态，国内我不是很了解


**你提到未来要用Amore引入LSM实现秒级可见，当前的10分钟级upsert满足不了什么业务场景？**

Iceberg 本身是为批处理与离线分析设计的，文件不可变，像一些实时风控与欺诈检测（金融 / 支付），电商 / 直播实时大屏与运营干预，物联网（IoT）设备状态监控与告警等场景它是支撑不了的，Amoro 的 Mixed-Iceberg 引入 LSM-Tree 结构，将 Iceberg 从 “批处理快照” 改造为 “实时可更新存储”，写入先落内存，秒级即可见，无需等待快照合并，内存满后刷写 SSTable（小文件），后台异步 Compaction 合并大文件，读写分离、互不阻塞，这样的话数据只要写入内存中下游就可见了，延迟至少低一个数量级。

**MinIO作为对象存储，在实际使用中遇到过哪些网络或性能瓶颈？**

在实际使用 MinIO 时，性能瓶颈通常不会出现在 MinIO 软件本身（因为它非常高效），而是更多地受限于硬件基础设施和配置策略，就比如我的个人集群中，受限于只有一块nvme硬盘，并由各个组件公用，导致io争抢，使Minio跑不到极速，通过我只有千兆网卡，如果同时有几个大任务在跑的话，会有带宽争抢。

**具体是如何用OpenMetadata做数据质量检查的？能举一个数据质量规则的例子吗？**

我对数据质量的使用并不多，主要集中的每天会跑一个任务对比前一天dwd层表的数据量是不是和ods层主表的相等，和sum值有否相等，如果不相等则发送邮件告警，由人工介入，不会阻塞下游任务的运行。

**Airflow DAG之间的血缘和OpenMetadata的血缘是如何关联的？**

Openmetadata的血缘关系都是基于拉取模型的，配置好Airflow的连接信息即可自动拉取了，比起Airflow dag的血缘，更重要的是Iceberg表的表级血缘和列级血缘，我在Openmetadata中配置好Trino的连接信息后，即可自动可视化血缘关系，一旦业务系统要求修改某个字段时，就可以通过血缘关系追溯到上游的表，防止遗漏。

**10分钟级的DAG和每日零点的DAG是如何协调的？如果10分钟任务积压了怎么办？**

我会在Airflow中配置依赖关系，因为每日零点的DAG属于大任务，容易和小任务抢锁，如果大任务运行时间超过10分钟，可能会导致大任务永远都无法运行成功，因此我会给分钟任务设置零点任务的依赖，需要等到大任务运行完后才能继续小任务，即使大任务运行失败了，也只是任务级别的依赖，成功一个则运行下游小任务，即使有大任务一直失败，也通过Airflow的配置策略使小任务继续运行，如果大任务运行时间比较久，小任务确实可能积压，我的方案是在Airflow中设置dag同时运行个数的限制，必须等前上一个周期的dag运行完成才可以继续下一个dag，直至成功。

### Spark部分
**请简述 Spark 的核心组件及其作用？**

答：简单来说，RDD 就是 Spark 中最基本的分布式内存数据抽象。 它有三个核心特点，第一是不可变：一旦创建就不能修改，只能通过转换生成新的 RDD。二是可分区：数据被切分成多份，分布在集群不同节点上并行计算。三是可容错：通过记录血缘关系，如果数据丢失可以自动恢复。 大家都知道编程语言中都有各自的数据结构，比如python中的list，java中的arrayList，Spark中的RDD和这个语言的数据结构类似，不过它在这些数据结构之上做了更高级的封装，使其具备分区，容错等属性。

**Spark和Mapredduce的区别是什么**

首先，MapReduce 每步都要落盘，Map 完写 HDFS，Reduce 再从 HDFS 读，IO 开销特别大。而 Spark 是基于内存的，中间结果可以存在内存里，只有内存不够或者遇到宽依赖才溢写到磁盘。 其次，Spark 的 DAG 优化器很智能。它会帮你把多个 Transformation 操作串起来，形成一个 Pipeline，一次性执行。不像 MapReduce，每个 Map-Reduce 阶段都要单独提交。 还有就是 API 友好很多。我平时主要用 DataFrame API，直接写 SQL 或者用链式调用，比 MapReduce 手写 Mapper、Reducer 简单多了，而且 Catalyst 优化器会自动帮你做谓词下推、列裁剪这些优化。 不过说实话，Spark 也不是银弹。如果数据量特别大，内存放不下，或者集群资源紧张，Spark 也可能频繁 Shuffle 溢写，这时候性能就会打折扣。所以关键还是要会调优。"

**Spark 中 RDD、DataFrame 和 Dataset 的区别是什么？**

首先是RDD，它是最底层的抽象，是分布式的 Java/Scala 对象集合。优点是细粒度控制，缺点是缺乏元数据信息，无法进行语义优化。 然后是DataFrame，它类似关系型数据库的表，带有 列名和类型。基于 Catalyst 优化器，执行效率通常高于 RDD。 最后是Dataset，它是DataFrame 的扩展，提供了类型安全的对象操作接口。在 Scala/Java 中常用，结合了 RDD 的类型安全和 DataFrame 的执行优化。

**什么是 DAG？Spark 为什么使用 DAG 而不是像 MapReduce 那样直接执行？**

DAG是有向无环图，Spark将作业转换为一系列阶段，每个阶段包含多个含依赖关系的task，形成DAG，之所以Spark使用DAG是因为和MapReduce不一样的是它是内存计算，大部分任务都可以在内存中直接计算，迭代计算速度非常快 然后是优化执行，dag调度去可以在运行前进行全局优化，如流水线执行、分区裁剪等，再然后是可以基于血缘进行容错，而非单纯依赖数据副本。

请描述 Spark 任务的提交流程（从 submit 到执行）

首先用户先提交spark-submit，然后driver启动，构建上下文，driver向k8s申请资源，然后k8s启动executor，driver将job按宽依赖划分成阶段，driver再将阶段划分成task，并分发给executor， executor执行任务，并将结果返回给Driver或写入存储。

**什么是宽依赖和窄依赖？它们对 Stage 划分有什么影响？**

窄依赖是父RDD的一个分区只被子RDD的一个分区使用（如map，filter，union）。允许流水线执行。 宽依赖是父RDD的一个分区被子RDD的多个分区使用（如groupByKey，reduceByKey，Join）。涉及shuffle操作。 stage划分的原则是遇到宽依赖是会切断，生成新的stage，窄依赖则合并到同一个stage中。

**请说说Spark的Shuffle机制演进与调优**

答：Spark的Shuffle实现随着版本的迭代发生了多次重大变更，主要目标是提高性能、减少内存占用并增强稳定性。 早期Spark用的是Hash Shuffle，每个MapTask 会为每个ReduceTask生成一个单独的文件，优点是不需要排序，会快一点，缺点是需要同时打开大量的缓冲区，虽然后来引入了一些机制合并文件，但依旧有些力不从心； 第二个阶段是Sort Shuffle（Spark2.0之后成为默认选项），通过每个MapTask最终只产生一个数据文件和一个索引文件解决了小文件问题，原理是数据先写入一个内存缓冲区中，当缓冲区满时，对数据进行排序并溢写到磁盘，最后将所有溢写文件合并成一个大的数据文件和一个索引文件，Reduce端通过索引文件去数据文件中读取属于自己的数据，优点是文件数量大幅减少，且支持在内存排序，有利于后续的聚合和排序操作，缺点是引入了排序，CPU开销略高于Hash Shuffle。 第三个阶段是Tungsten-Sort，这是对Sort Shuffle的深度优化，主要利用堆外内存和二进制格式处理数据，优点是可以大幅减少JVM的内存占用和CPU开销，缺点是有触发条件，大概就是需要使用Spark SQL自带的序列化器和KryoSeializer，且没有聚合操作。 至于Shuffle的调优，核心思想就是减少Shuffle数据量或者减少Shuffle次数或者优化Shuffle过程的资源使用. 你可以先过滤掉不必要的数据，其次在Join操作中，如果一个表比较小，可以使用广播变量将小表广播到所有Executor，这样可以避免shuffle。 其次就是参数的调优了，并行度过高会导致大量的小文件和调度开销，过低会导致单个Task处理数据量过大，一般设置为总核数的2-3倍即可。 内存缓冲区默认是32KB，如果有足够的内存，可以适当调大，减少磁盘溢写次数，reduce端拉取数据时也有个内存缓冲区大小，调大它可以减少网络拉取次数，但会增加内存压力 也可以开启shuffle文件压缩，虽然会增加cpu负担，但能显著减少磁盘和网络io 还有就是对数据倾斜的调优，大致可以通过提升并行度，广播小表，给倾斜key加盐值的方式减小shuffle的压力

**在数据倾斜场景下，除了常规的加盐（Salting），如何利用 Spark 3.x 的 AQE (Adaptive Query Execution) 动态合并小分区或动态调整倾斜连接策略？请描述其内部触发机制。**

Spark的AQE是Spark引入的新特性嘛，它利用运行时的统计信息动态优化查询计划，从而解决了传统Spark SQL在编译阶段无法感知数据分布的痛点。原理主要是通过动态合并Shuffle分区，及动态处理倾斜Join来做的。 首先是动态合并Shuffle分区，在没有AQE的情况下，我们需要手动设置分区数，如果设置得太大，会产生大量的小文件，导致大量随机小IO和网络开销；如果设置得太小，单个Task处理数据量过大，容易OOM，引入AQE以后，AQE会在Shuffle阶段结束后，根据实际运行时的数据分布，动态地将相邻的小分区合并成较大的分区，以减少task数量，至于原理，是spark在driver端收集了每个shuffle map stage输出的各个分区的统计信息，根据AQE中设置的参数来确定合并后的目标大小，内部走的是一个判断逻辑，首先从第一个分区开始遍历，一旦大小接近参数值，就将其标记成一个合并后的新分区，然后就是如果单个分区大小本身就超过了目标大小，它会被保留为一个独立的分区，最后是累加后的总和远小于目标大小，但已经到了最后一个分区，也会强制合并，然后就是根据合并后的分区映射关系，动态重写后续stage的物理执行计划。 然后是动态处理倾斜key，这是AQE针对数据倾斜的方案，在没有AQE的情况下，我们需要手动编写加盐值的逻辑，非常繁琐而且还容易出错，AQE可以在运行时自动检测倾斜并拆分处理，原理就是，AQE可以识别到分区数据量是不是大于了配置的参数，一旦检测到倾斜分区，它就会根据比例将大的分区拆成N个小分区，把非倾斜侧的需要Join的分区复制N份，广播拆分的小分区中，再执行Join

**如果开启 AQE 后，某些作业反而变慢了，可能的原因是什么？如何排查？**

开启AQE后作业变慢，通常是因 AQE的动态优化策略与实际的数据分布、集群资源状态或代码逻辑发生了冲突。虽然AQE在大多数情况下能提升性能，但在特定场景下，它的副作用可能会超过其带来的收益。 一些常见原因有动态合并shuffle分区导致并行度下降，导致作业运行时间变长，CPU空闲，然后是动态倾斜Join的拆分开销过大，他会复制非倾斜侧的分区数据嘛，如果复制得比较多也会产生大量的网络传输和磁盘io，然后就是运行时统计信息的收集和规划延迟，再stage比较多的情况下，每个stage都会由driver收集统计信息，累计的调度延时也不可忽视。 排查思路肯定就是检查Spark UI嘛，你可以找到哪个页面的stage，观察一下被AQE修改的部分，观察shuffle数据量是否异常，task的分布是否不均，如果所有task都变慢了，可能是被过度合并了，如果是长尾问题，可能是倾斜处理失效或引入了新的倾斜，如果shuffle数据异常，意味着join的策略选择错误，本来该广播的却选择了shuffle。 然后就是调整AQE参数，尝试让其恢复正常，分区合并的话就要尝试让其合并后的分区数达到最佳实践的数量，或者调整分区大小以保留更多分区，然后就是手动调整join策略，直到恢复正常为止，还有一个点就是要检查driver的资源，如果driver资源紧张，可能会导致规划阶段耗时过长

**解释 Spark 统一内存管理模型中 Storage Memory 和 Execution Memory 的动态借用机制。**

Spark在1.6版本后引入了统一内存管理模型，旨在解决以前静态内存划分导致资源利用率不高的问题，在统一内存管理模型中，存储内存和和执行内存共享同一块堆内内存区域，这两块区域不是静态固定的，而是可以相互借用的，首先，这个

**当发生 Spill to Disk 时，对性能的具体影响有哪些？在什么配置下会优先选择抛 OutOfMemoryError 而不是 Spill？**

我先说一下溢写到磁盘对性能的影响，溢写的本质是将高速的内存读写转化为低速的磁盘io， 首先就是磁盘io开销加剧，spark虽然在写的时候会尽量顺序写入文件，但在shuffle read阶段，reduce需要从多个map端拉取文件，这不可避免的引入了随机io，比单纯的内存交换慢几个数量级，其次时数据在写入时需要序列化，读取时需要反序列化，对于复杂对象，这个过程非常消耗cpu 然后就是额外的排序与合并开销，当内存缓冲区满了触发溢写时，spark还会对内存中的数据进行排序，如果代码逻辑里面还有排序，还需要再内部进行二次排序，然后在task结束前，如果产生了多个溢写文件，spark还需要将这些文件和内存中的剩余数据一起进行归并排序，合并成一个大文件，文件越多，开销越大。 然后就是会拉长GC的时间，在溢写的过程中，为了管理磁盘上的数据指针，spark会频繁的创建辅助对象，频繁的内存申请和释放会导致堆内存碎片化，导致频繁GC，拖慢整个作业。 让然后就是溢写的数据是通过网络进行交换的，会导致网络请求的次数会增加。

然后我再说一下什么情况下会优先OOM而不是溢写，通过情况下，堆外内存溢出是最常见的，堆外内存不受JVM管理，且通常不直接支持像堆内内存那样灵活的溢写机制，就会OOM 然后是算子逻辑导致的无法spill，有些特定的数据结构不支持溢写，或者代价极高，spark容易崩溃，比如说广播变量，广播变量设计初衷就是希望能完全驻留内存的，溢写到磁盘就违背了初衷了， 然后就是加载数据那里，数据是放在堆内存的局部变量中，不属于spark统一内存管理的范畴，spark的内存管理器无法感知这部分内存，也就没法溢写了

**请描述 Catalyst 优化器的四个阶段。**

第一个阶段，分析阶段，就是解析sql中的引用，就是利用spark的catalog来查找表名、列名，它会生成一个解析后的逻辑计划，如果在这个过程中发现引用不存在或者类型不匹配，查询会直接报错； 第二个阶段，逻辑优化阶段，核心是生成一个优化后的逻辑计划，这个阶段不涉及具体的物理实现（比如说用Hash Join还是Sort Merge Join），只关注逻辑上的等价变换，具体就是运用一些规则，比如说谓词下推，列剪枝，常量折叠，布尔简化，消除冗余之类的操作； 第三个阶段，物理计划阶段，将逻辑计划转化为物理上怎么做，一个逻辑计划一般会对应多个物理计划，Catalyst会根据策略选择一个最佳的物理计划，具体就是选择算子具体的实现方式，比如说选择join的方式。 第四个阶段就是代码生成阶段，这就是Spark SQL性能优于hive的关键，通过迭代器模式处理数据，这里就会使用全阶段代码生成，将多个物理算子合并成一个单一的java函数，同时在运行时动态生成java字节码，具体就是把物理计划树压成一个个RDD操作。

**Whole-Stage Code Generation 是如何工作的？它解决了传统火山模型（Volcano Model）的什么瓶颈？**

首先我说说传统的火山模型嘛，火山模型的工作方式会把查询计划表示为一个算子树，每个算子都实现了一个统一的接口，比如说open()，next()，close等，父算子就调用子算子的next()方法获取一行数据，子算子处理完数据后返回给父算子，父算子处理改行数据，再次调用next()，如此往复，优点是简介易扩展，缺点就是在性能上存在严重的缺陷，主要源于虚函数调用和内存访问。 首先虚函数调用的开销来源于每处理一行数据，都需要在多个算子之间进行函数调用，如果查询计划有N层深度，处理一行数据就需要N次函数调用，在JVM中，这些调用无法被编译器有效管理，导致CPU指令流水线频繁中断 然后就是CPU缓存局部性差，数据在各个算子的函数栈中传递，导致CPU缓存命中率低 然后就是全阶段代码生成的原理，它的核心机制就是算子融合，它不会为每个算子单独生成代码，而是寻找物理计划中适合融合的算子链，将它们压扁成一个大的代码块。

**如果在执行计划中看到 Exchange 算子非常多，通常意味着什么？如何通过 repartition 或 coalesce 优化？**

Exchange算子多就说明在频繁的shuffle，会导致大量的网络传输，也会加重内存和磁盘压力。 repartition 或 coalesce 的核心优化目标肯定就是减少不必要的Shuffle，或者合并小分区以提高效率嘛 首先repartition会触发shuffle，当出现数据倾斜时，可以增加分区数，把数据再打散一点，然后就是可以提高并行度，如果再join之前按照join的key进行了repartition，也可以避免Spark在join时自动进行额外的shuffle，或者至少让shuffle按照预期进行。 coalesce不会触发shuffle，用于减少分区数，适用于那种过滤了大量数据，但分区数没变，导致产生了大量小文件或空任务，此时就能使用coalesce减少分区数，还有就是在写入数据时，可以减少一点分区数，减少最终产生的小文件数量，然后就是可以提升shuffle read的性能。

**在批处理中，如果 Driver 端 TaskScheduler 发现大量 Task 失败，重试机制是怎样的？如何区分是数据问题还是资源问题导致的反复重试？**

Spark的容错机制主要依赖于task的重试。spark有很多判断task是否失败的方式，第一时executor通过心跳机制向Driver汇报Task的状态，如果Task执行异常，或者Executor超时未汇报心跳，Driver会判定该Task失败了，然后Driver会将该Task放到调度队列重新调度，默认配置好像是允许失败4次，同时这里还有黑名单机制，为了防止坏节点导致task反复失败，失败太多的executor在会被加入黑名单，后续task不会调度到该节点。然后是如果失败了4次，spark会标记整个stage为失败，Driver会尝试重新提交stage，和task类似，如果stage失败达到上限，会标记整个作业都失败。 至于区分问题的方式，一般都还是看Spark WEB UI的指标，数据问题的话一般会表现为特定的task失败，且失败具备确定性，不管重试多少次，只要处理到该数据，就会报错，具体表现一般有数据倾斜，大部分task都完成了，极少数task还在运行，然后是特定脏数据异常，只有处理到某条数据才报错，报错信息也包含具体的业务逻辑异常 资源问题一般表现为随机性失败，失败与具体的数据内容无关，一般就是OOM，这种就是数据太大装不下或者大量数据被塞到一个executor里面了，然后是报错信息有什么lost之类的，一般就意味着机器宕机，网络抖动、gc超时之类的。

## 数仓部分
**请介绍一下数据仓库的分层架构，以及你在项目中的实践**
  
在我之前的项目中，我们采用的是标准的五层架构：ODS、DIM、DWD、DWS和ADS。
简单来说，ODS层就是贴源层，源系统的数据原封不动地同步过来，不做太多处理。这样做的好处是，如果源系统数据结构变了，我们只需要改ODS的同步逻辑，不会影响下游。
DIM层是维度层，统一管理用户、商品、时间这些维度数据。
DWD层是我最关注的，这是明细事实层。我们有个很重要的原则——一个业务过程一张表。比如下单是一张表，支付是另一张表，不会混在一起。而且我们规定一个指标只能归属一张表，不允许同一指标在多张表里重复定义。之前遇到过这个问题，"支付金额"这个字段在两张表里都有，但命名不一样，一个叫payment_amt，一个叫pay_amount，下游用的时候经常搞混，所以需要在命名上区分它们
DWS层是汇总层，也是指标的统一出口。所有跨业务过程的指标组装都在这一层完成。比如说"用户下单后24小时内的支付率"，这个指标涉及下单和支付两个过程，就必须在DWS层做，DWD层不碰这种跨过程的指标。
ADS层就是看业务需求来灵活建设了
我们还有个硬性规定：禁止跨层取数。DWD必须从ODS取数，不能直接连源数据库。这样数据血缘很清晰，出了问题也容易排查。

**DWD层的事实表有哪几种？**

事务事实表：一行代表一个具体的业务操作，通常只插入，不更新，是最常用的
周期快照事实表：在固定时间间隔对业务状态进行快照，一行代表某个时间点（天/周/月）的汇总状态，示例：每日库存快照表、每月账户余额表、每天订单状态汇总表
累积快照事实表：跟踪业务实体的完整生命周期，记录关键里程碑，示例：订单生命周期表（包含下单时间、支付时间、发货时间、签收时间等）
无事实的事实表：只有维度外键，没有度量值，示例：用户签到表、活动参与记录表

**你的DWS层是怎么设计的？有什么原则？**

DWS层我们定位得很明确——它是数仓的指标统一出口，下游只能从这一层拿指标。
设计原则有几个：
第一，面向主题域建表，不面向报表建表。 比如说我们建"用户行为日汇总表"，而不是"XX报表数据表"。因为报表会变，但主题是相对稳定的。
第二，派生指标有四个要素：原子指标、修饰词、时间周期和聚合维度。 比如"近7日北京地区用户下单数"——"下单数"是原子指标，"北京地区"是修饰词，"近7日"是时间周期，"用户"是聚合维度。这四个要素组合起来才是一个完整的派生指标。
第三，不同时间粒度分表存储。 日表和月表分开建，不会混在一张表里。这样查询的时候分区裁剪更清晰，性能也更好。
第四，适度冗余。 有些高频筛选的维度我们会直接冗余到DWS表里，避免频繁JOIN。当然这个要权衡，不能为了冗余而冗余。
我们有个硬性规定：DWD层不做跨过程指标拼接。比如订单和售后的关联分析，必须在DWS层做，DWD层只负责把各自的事实表建好。

**维度建模中，什么是事实表的"粒度"？为什么它很重要？**

粒度是事实表中一行数据所代表的业务含义，简单说就是**"一行数据代表什么"**。
比如说订单事实表，如果粒度是"每个订单项"，那么一个订单如果有3个商品，就会有3行记录。如果粒度是"每个订单"，那不管有多少商品，都只有1行。
为什么重要？ 因为粒度决定了：
第一，数据能不能下钻分析。 如果粒度是"订单级别"，你就没法分析单个商品的销售情况了。反之，如果粒度是"订单项级别"，你可以向上聚合到订单级别，但反过来不行。所以我们的原则是DWD层尽量保持最细粒度，汇总的事交给DWS层做。
第二，指标计算是否正确。 比如计算GMV，如果粒度搞错了，可能会重复计算。我们之前就踩过坑，有个表粒度是"用户-商品-天"，但计算销售额的时候直接SUM，结果同一个订单被算了多次。
第三，存储和性能的权衡。 粒度越细，数据量越大，但灵活性越高。我们DWD层用细粒度，DWS层根据分析需求做适当的汇总。

**数据仓库的模型设计方法有哪些？你更倾向于哪种？**

第一种是范式建模（ER模型）。 就是传统数据库的第三范式设计，消除数据冗余。优点是数据一致性好，存储高效。缺点是查询复杂，JOIN太多，性能差。这种在传统数仓 like Teradata用得比较多。
第二种是维度建模。 就是Fact+Dimension的星型模型，也是我主要用的方法。优点是查询性能好，业务理解门槛低。缺点是有一定冗余，但随着存储成本降低，这个已经不是大问题了。
一般主流都是用的维度建模，原因是
业务友好：星型模型很直观，业务人员容易理解性能优秀，减少了JOIN，查询效率高
生态成熟：工具链完善，从建模到可视化的支持都很好
当然，我们也不是完全不用范式。ODS层其实就比较接近范式模型，保持和源系统一致。到了DWD层才转为维度模型。实际项目中往往是混合使用，不是非黑即白。

**数仓中如何处理缓慢变化维？你们的实践是什么？**

缓慢变化维（SCD）是维度建模里的经典问题。比如用户的收货地址变了、商品的价格调整了，这些维度属性不是经常变，但变了之后怎么处理？
常见的处理方式：
SCD Type 1（覆盖更新）：直接UPDATE，不保留历史。适用于纠错场景，比如用户手机号录错了，直接改过来就行，不需要保留错误版本。
SCD Type 2（新增行）：每次变化插入新记录，用生效时间、失效时间和当前标识来管理。适用于需要完整历史追溯的场景，比如分析用户在不同城市的消费行为变化。
SCD Type 3（新增列）：加一个"上次值"字段，只能保留最近一次变化。用得比较少，因为扩展性差。
我们的实践：
对于DIM层，大部分场景用SCD Type 2。但如果用Iceberg，我们有更简单的方案——时间旅行。 直接查询某个时间点的快照，这样就不需要维护复杂的SCD Type 2结构了，大大简化了ETL逻辑。
选择建议：
如果用的是传统Hive/MySQL，SCD Type 2是标准做法
如果用Iceberg等数据湖技术，优先考虑时间旅行
关键是看业务是否需要追溯历史，不需要的话Type 1就够了

**数仓建设中，遇到过最大的挑战是什么？怎么解决的？**

最大的挑战应该是指标口径不一致导致的数据信任危机。
当时的情况是，公司高管看报表，发现财务部门的数据、运营部门的数据、数仓的数据对不上。同样是"月销售额"，三个数字都不一样。高管直接质疑："你们的数据到底哪个是对的？"
这个问题很严重，因为一旦失去信任，数仓的价值就没了。
根本原因：
各部门自己定义指标，没有统一管理
数仓早期为了快速响应需求，没有严格执行规范
源系统本身数据质量就有问题（比如退款没及时同步）
解决过程：
第一步：止血。 先盘点所有对外报表，标注每个指标的来源和计算逻辑，找出差异点。
第二步：统一口径。 拉着业务、财务、运营开会，逐一确认每个指标的官方定义。这个过程很痛苦，因为各方都有自己的"道理"，最后只能高管拍板。
第三步：技术重构。
建立指标字典，明确每个指标的归属层和计算逻辑
DWD层严格遵循"一个指标一张表"
DWS层作为唯一出口，下游只能用DWS的数据
废弃那些不规范的临时表
第四步：建立流程。
新增指标必须走评审流程
指标变更要通知所有下游
定期做数据对账
第五步：重建信任。
主动和业务团队对齐数据
建立数据质量报告，定期发布
有问题及时沟通，不藏着
经验教训：
规范必须严格执行，不能因为赶工期就妥协
指标管理要趁早，越到后面越难改
沟通比技术重要，很多问题不是技术解决不了，而是各方利益不一致
透明度很重要，数据来源、计算逻辑要公开，接受监督

